## Exercises

1. Implement batch gradient descent, stochastic gradient descent (SGD), and mini-batch gradient descent. Compare convergence speed on the same dataset. Which converges fastest? Which has the smoothest cost curve?
2. Generate data from a cubic function (y = ax^3 + bx^2 + cx + d + noise). Fit polynomials of degree 1, 3, and 10. Compare training R^2 and test R^2. At what degree does overfitting become obvious?
3. Implement Lasso regression (L1 regularization: penalty = alpha * sum(|w_i|)). Train on the multi-feature housing data. Compare which weights go to zero vs Ridge. Why does L1 produce sparse solutions while L2 does not?


### 1 Ans

In [60]:
class LinearRegression:
    def __init__(self,n_features=1,learning_rate=0.01):
        self.weights=[0.00]*n_features
        self.bias=0.00
        self.lr=learning_rate
        self.n_features=n_features
        self.cost_history=[]

    def predict_single(self,X):
        # prediction for single feature
        # X -> vector 
        y = 0
        for xi,wi in zip(X,self.weights):
            y+= xi * wi

        y+=self.bias
        return y

    def predict(self,X):
        predicted_arr=[]

        for x in X:
            predicted_arr.append(self.predict_single(x))
        return predicted_arr

    def cost(self,X,y):
        # y is the vector and X is the matrix so have to calculate the mse
        cost=0
        n=len(y)
        y_predicted=self.predict(X)

        for pred,actual in zip(y_predicted,y):
            cost+=(pred-actual)**2

        cost = (1/n)*cost

        return cost 
    def r_squared(self, X, y):
            predictions = self.predict(X)
            y_mean = sum(y) / len(y)
            ss_res = sum((actual - pred) ** 2 for actual, pred in zip(y, predictions))
            ss_tot = sum((actual - y_mean) ** 2 for actual in y)
            return 1 - (ss_res / ss_tot)
            


    def fit(self,X,y,epochs=100,print_at=100):

        n=len(y)
        
        for epoch in range(epochs):
            y_predicted = self.predict(X)
            for j in range(self.n_features):
                # grad_w=(2/n)*sum(pred-actual for pred ,actual in zip(y_predicted,y)
                
                grad_w=0
                
                for pred,actual,i in zip(y_predicted,y,range(n)):
                    
                    grad_w+=((pred-actual)*X[i][j])
                    
                self.weights[j]-=(2/n)*(self.lr*grad_w)
            
            grad_b=0            
            for pred,actual in zip(y_predicted,y):
                grad_b+=(pred-actual)
            
            self.bias-=(2/n)*self.lr*grad_b
            
            # cost=self.cost(X,y)
            # self.cost_history.append(cost)
            # print(f"epoch {epoch} : grad_w: {self.weights} grad_b: {self.bias}")
            cost = self.cost(X, y)
            r2 = self.r_squared(X, y)
            
            if epoch % print_at == 0:
                print(
                    f"\nEpoch {epoch+1:4d}/{epochs}"
                    f"\n{'─'*62}"
                    f"\nLoss      : {cost:.6f}"
                    f"\nR² Score  : {r2:.6f}"
                    f"\nWeights   : {[round(w,4) for w in self.weights]}"
                    f"\nBias      : {self.bias:.4f}"
                )
        return self

        def fit_minibatch(self, X, y, batch_size=32, epochs=100, print_at=100):

            n = len(y)
        
            for epoch in range(epochs):
        
                for start in range(0, n, batch_size):
        
                    end = min(start + batch_size, n)
        
                    X_batch = X[start:end]
                    y_batch = y[start:end]
        
                    predictions = [self.predict_single(x) for x in X_batch]
        
                    m = len(y_batch)
        
                    # weights
                    for j in range(self.n_features):
        
                        grad_w = 0
        
                        for pred, actual, x in zip(predictions, y_batch, X_batch):
                            grad_w += (pred - actual) * x[j]
        
                        self.weights[j] -= self.lr * (2 / m) * grad_w
        
                    # bias
                    grad_b = 0
        
                    for pred, actual in zip(predictions, y_batch):
                        grad_b += pred - actual
        
                    self.bias -= self.lr * (2 / m) * grad_b
        
                cost = self.cost(X, y)
                r2 = self.r_squared(X, y)
        
                if epoch % print_at == 0:
                    print(
                        f"\nEpoch {epoch+1:4d}/{epochs}"
                        f"\n{'─'*62}"
                        f"\nLoss      : {cost:.6f}"
                        f"\nR² Score  : {r2:.6f}"
                        f"\nWeights   : {[round(w,4) for w in self.weights]}"
                        f"\nBias      : {self.bias:.4f}"
                    )

        def fit_sgd(self, X, y, epochs=100, print_at=100):

            n = len(y)

            for epoch in range(epochs):
        
                for i in range(n):
        
                    pred = self.predict_single(X[i])
                    error = pred - y[i]
        
                    # update weights
                    for j in range(self.n_features):
                        grad_w = 2 * error * X[i][j]
                        self.weights[j] -= self.lr * grad_w
        
                    # update bias
                    grad_b = 2 * error
                    self.bias -= self.lr * grad_b
        
                cost = self.cost(X, y)
                r2 = self.r_squared(X, y)
        
                if epoch % print_at == 0:
                    print(
                        f"\nEpoch {epoch+1:4d}/{epochs}"
                        f"\n{'─'*62}"
                        f"\nLoss      : {cost:.6f}"
                        f"\nR² Score  : {r2:.6f}"
                        f"\nWeights   : {[round(w,4) for w in self.weights]}"
                        f"\nBias      : {self.bias:.4f}"
                    )

                        
            
            
        
            

    

In [63]:
X = [
    [2, 70, 3],
    [3, 75, 5],
    [4, 80, 6],
    [5, 85, 7],
    [6, 90, 8],
    [7, 92, 9],
    [8, 95, 10],
    [9, 97, 10],
    [10, 98, 10],
    [11, 100, 10],
]

y = [
    40,
    48,
    56,
    64,
    72,
    80,
    88,
    92,
    96,
    100,
]


X_test = [
    [2.5, 72, 4],
    [3.5, 78, 5],
    [4.5, 83, 6],
    [5.5, 87, 7],
    [6.5, 91, 8],
    [7.5, 94, 9],
    [8.5, 96, 10],
    [9.5, 99, 10],
]

# Expected Marks
y_test = [
    44,
    52,
    60,
    68,
    76,
    84,
    90,
    98,
]

In [67]:
# Training

model = LinearRegression(
    n_features=3,
    learning_rate=0.0001
)

# Batch Gradient Descent
model.fit(X, y, epochs=1000)

# Stochastic Gradient Descent
model.fit_sgd(X, y, epochs=1000)

# Mini-Batch Gradient Descent
model.fit_minibatch(
    X,
    y,
    batch_size=32,
    epochs=1000
)


Epoch    1/1000
──────────────────────────────────────────────────────────────
Loss      : 2159.711754
R² Score  : -4.434057
Weights   : [0.107, 1.337, 0.124]
Bias      : 0.0147

Epoch  101/1000
──────────────────────────────────────────────────────────────
Loss      : 102.891861
R² Score  : 0.741113
Weights   : [0.5281, 0.7751, 0.3777]
Bias      : -0.0149

Epoch  201/1000
──────────────────────────────────────────────────────────────
Loss      : 79.409463
R² Score  : 0.800198
Weights   : [0.9313, 0.7208, 0.6398]
Bias      : -0.0360

Epoch  301/1000
──────────────────────────────────────────────────────────────
Loss      : 61.320631
R² Score  : 0.845711
Weights   : [1.2853, 0.6733, 0.8695]
Bias      : -0.0545

Epoch  401/1000
──────────────────────────────────────────────────────────────
Loss      : 47.386510
R² Score  : 0.880771
Weights   : [1.5963, 0.6315, 1.0709]
Bias      : -0.0707

Epoch  501/1000
──────────────────────────────────────────────────────────────
Loss      : 36.65280

AttributeError: 'LinearRegression' object has no attribute 'fit_sgd'

,area,bedrooms,bathrooms,stories,parking
0,1.440625,4,2,3,2
1,1.739623,4,4,4,3
2,1.933777,3,2,2,2
3,1.456158,4,2,2,3
4,1.440625,4,1,2,2
...,...,...,...,...,...
540,0.582463,2,1,1,2
541,0.465970,3,1,1,0
542,0.702839,2,1,1,0
543,0.564989,3,1,1,0


### 2. Ans

In [68]:
import numpy as np

np.random.seed(42)

# Generate data
X = np.linspace(-3, 3, 100)

y = (
    2*X**3
    - 4*X**2
    + 3*X
    + 5
    + np.random.normal(0, 5, size=len(X))
)

In [69]:
X

array([-3.        , -2.93939394, -2.87878788, -2.81818182, -2.75757576,
       -2.6969697 , -2.63636364, -2.57575758, -2.51515152, -2.45454545,
       -2.39393939, -2.33333333, -2.27272727, -2.21212121, -2.15151515,
       -2.09090909, -2.03030303, -1.96969697, -1.90909091, -1.84848485,
       -1.78787879, -1.72727273, -1.66666667, -1.60606061, -1.54545455,
       -1.48484848, -1.42424242, -1.36363636, -1.3030303 , -1.24242424,
       -1.18181818, -1.12121212, -1.06060606, -1.        , -0.93939394,
       -0.87878788, -0.81818182, -0.75757576, -0.6969697 , -0.63636364,
       -0.57575758, -0.51515152, -0.45454545, -0.39393939, -0.33333333,
       -0.27272727, -0.21212121, -0.15151515, -0.09090909, -0.03030303,
        0.03030303,  0.09090909,  0.15151515,  0.21212121,  0.27272727,
        0.33333333,  0.39393939,  0.45454545,  0.51515152,  0.57575758,
        0.63636364,  0.6969697 ,  0.75757576,  0.81818182,  0.87878788,
        0.93939394,  1.        ,  1.06060606,  1.12121212,  1.18

In [70]:
y

array([-91.51642923, -89.86259345, -81.26304603, -72.37282968,
       -76.79883822, -72.58977931, -59.46231308, -59.60607426,
       -62.01840735, -53.32626815, -54.86176387, -51.51383395,
       -44.74811501, -52.42662884, -48.51400675, -39.85426259,
       -39.38193394, -30.14036976, -33.76175824, -33.90671983,
       -17.25138161, -23.55112042, -20.03272935, -25.5450894 ,
       -19.2944155 , -14.2665463 , -18.91961713,  -9.72181043,
       -13.12863519, -10.19587031, -10.4420417 ,   3.05029929,
        -5.13496532,  -9.28855464,   1.10674187,  -8.18697418,
        -0.18333041, -10.23633932,  -6.35203579,   1.93997861,
         5.25734946,   2.97643959,   2.04384724,   1.56964083,
        -3.91112847,   0.24450548,   1.86137191,   9.73228169,
         6.41080369,  -3.90983862,   6.7077115 ,   3.31576065,
         1.98506471,   8.53385241,  10.71622977,  10.28603023,
         1.48724711,   4.1789569 ,   7.4136703 ,  10.66073479,
         3.40878696,   4.89667659,   0.3149456 ,  -0.10

In [71]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

X_train = X_train.reshape(-1,1)
X_test = X_test.reshape(-1,1)

In [72]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [74]:
degrees = [1,2,3]

for degree in degrees:
    model = make_pipeline(
        PolynomialFeatures(degree),
        LinearRegression()
    )

    model.fit(X_train,y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)

    print(
        f"Degree {degree}: "
        f"Train R² = {train_r2:.4f}, "
        f"Test R² = {test_r2:.4f}"
    )

Degree 1: Train R² = 0.7759, Test R² = 0.7183
Degree 2: Train R² = 0.9044, Test R² = 0.9068
Degree 3: Train R² = 0.9731, Test R² = 0.9839


### 3. Ans

In [10]:
class RidgeRegression:
    def __init__(self, n_features, learning_rate=0.01, alpha=1.0):
        self.weights = [0.0] * n_features
        self.bias = 0.0
        self.lr = learning_rate
        self.alpha = alpha

    def predict_single(self, x):
        return sum(w * xi for w, xi in zip(self.weights, x)) + self.bias

    def predict(self, X):
        return [self.predict_single(x) for x in X]

    def fit(self, X, y, epochs=1000, print_every=200):
        n = len(y)
        n_features = len(X[0])
        for epoch in range(epochs):
            predictions = self.predict(X)
            errors = [pred - actual for pred, actual in zip(predictions, y)]
            mse = sum(e ** 2 for e in errors) / n
            reg_term = self.alpha * sum(w ** 2 for w in self.weights)
            cost = mse + reg_term
            for j in range(n_features):
                grad = (2 / n) * sum(errors[i] * X[i][j] for i in range(n))
                grad += 2 * self.alpha * self.weights[j]
                self.weights[j] -= self.lr * grad
            grad_b = (2 / n) * sum(errors)
            self.bias -= self.lr * grad_b
            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Cost: {cost:.4f} | L2 penalty: {reg_term:.4f}")
        return self


In [15]:
class LassoRegression:
    def __init__(self, n_features, learning_rate=0.001, alpha=0.1):
        self.weights = [0.0] * n_features
        self.bias = 0.0
        self.lr = learning_rate
        self.alpha = alpha

    def predict_single(self, x):
        return sum(w * xi for w, xi in zip(self.weights, x)) + self.bias

    def predict(self, X):
        return [self.predict_single(x) for x in X]

    def fit(self, X, y, epochs=1000, print_every=200):
        n = len(y)
        n_features = len(X[0])

        for epoch in range(epochs):
            predictions = self.predict(X)
            errors = [pred - actual for pred, actual in zip(predictions, y)]

            mse = sum(e ** 2 for e in errors) / n
            reg_term = self.alpha * sum(abs(w) for w in self.weights)
            cost = mse + reg_term

            for j in range(n_features):

                # MSE gradient
                grad = (2 / n) * sum(errors[i] * X[i][j] for i in range(n))

                # L1 gradient (subgradient)
                if self.weights[j] > 0:
                    grad += self.alpha
                elif self.weights[j] < 0:
                    grad -= self.alpha
                # if weight == 0: add nothing

                self.weights[j] -= self.lr * grad

            # Bias update
            grad_b = (2 / n) * sum(errors)
            self.bias -= self.lr * grad_b

            if epoch % print_every == 0:
                print(
                    f"Epoch {epoch:4d} | "
                    f"Cost: {cost:.4f} | "
                    f"L1 penalty: {reg_term:.4f}"
                )

        return self

In [16]:
# housing data sets
X=[
    [1200, 2, 2, 15, 8],
    [1500, 3, 2, 10, 6],
    [1800, 3, 3, 8, 5],
    [2100, 4, 3, 5, 4],
    [2500, 4, 4, 3, 3],
    [2800, 5, 4, 2, 2],
    [3200, 5, 5, 1, 1],
    [1400, 3, 2, 20, 10],
    [1700, 3, 2, 12, 7],
    [2300, 4, 3, 6, 4]
]
y=[220, 290, 360, 450, 580, 690, 850, 210, 310, 510]

In [17]:
for arr in X:
    arr[0]=arr[0]/100

In [18]:
X

[[12.0, 2, 2, 15, 8],
 [15.0, 3, 2, 10, 6],
 [18.0, 3, 3, 8, 5],
 [21.0, 4, 3, 5, 4],
 [25.0, 4, 4, 3, 3],
 [28.0, 5, 4, 2, 2],
 [32.0, 5, 5, 1, 1],
 [14.0, 3, 2, 20, 10],
 [17.0, 3, 2, 12, 7],
 [23.0, 4, 3, 6, 4]]

In [19]:
lasso_model=LassoRegression(5,alpha=0.1)

In [20]:
lasso_model.fit(X,y)

Epoch    0 | Cost: 239990.0000 | L1 penalty: 0.0000
Epoch  200 | Cost: 717.4918 | L1 penalty: 4.1945
Epoch  400 | Cost: 649.9463 | L1 penalty: 4.3390
Epoch  600 | Cost: 591.3147 | L1 penalty: 4.4728
Epoch  800 | Cost: 540.6506 | L1 penalty: 4.7153


In [21]:
lasso_model.weights

[24.729184511736744,
 -1.4695557732793796,
 8.250897758072343,
 0.47078733401621764,
 -16.123433810580117]

In [24]:
ridge_model=RidgeRegression(5,alpha=0.1,learning_rate=0.001)

In [26]:
ridge_model.fit(X,y)

  Epoch    0 | Cost: 602.5336 | L2 penalty: 89.3677
  Epoch  200 | Cost: 577.5336 | L2 penalty: 96.7725
  Epoch  400 | Cost: 557.4631 | L2 penalty: 104.2295
  Epoch  600 | Cost: 541.3473 | L2 penalty: 111.5687
  Epoch  800 | Cost: 528.4043 | L2 penalty: 118.6702


In [27]:
ridge_model.weights

[25.09323070771741,
 -3.761548250889531,
 10.264009505950234,
 3.662395054240577,
 -22.17985434573045]

### Lasso Regression (L1 Regularization)

### Objective Function

\[
\text{Loss} = \text{MSE} + \alpha \sum |w_i|
\]

### Training Progress

```
Epoch    0 | Cost: 239990.0000 | L1 penalty: 0.0000
Epoch  200 | Cost: 717.4918 | L1 penalty: 4.1945
Epoch  400 | Cost: 649.9463 | L1 penalty: 4.3390
Epoch  600 | Cost: 591.3147 | L1 penalty: 4.4728
Epoch  800 | Cost: 540.6506 | L1 penalty: 4.7153
```

### Learned Weights

| Feature | Weight |
|---------|--------:|
| Size | 24.7292 |
| Bedrooms | -1.4696 |
| Bathrooms | 8.2509 |
| Age | 0.4708 |
| Distance | -16.1234 |

---

## Ridge Regression (L2 Regularization)

### Objective Function

\[
\text{Loss} = \text{MSE} + \alpha \sum w_i^2
\]

### Training Progress

```
Epoch    0 | Cost: 602.5336 | L2 penalty: 89.3677
Epoch  200 | Cost: 577.5336 | L2 penalty: 96.7725
Epoch  400 | Cost: 557.4631 | L2 penalty: 104.2295
Epoch  600 | Cost: 541.3473 | L2 penalty: 111.5687
Epoch  800 | Cost: 528.4043 | L2 penalty: 118.6702
```

### Learned Weights

| Feature | Weight |
|---------|--------:|
| Size | 25.0932 |
| Bedrooms | -3.7615 |
| Bathrooms | 10.2640 |
| Age | 3.6624 |
| Distance | -22.1799 |

---

## Comparison

| Feature | Lasso | Ridge |
|---------|-------:|-------:|
| Size | 24.7292 | 25.0932 |
| Bedrooms | -1.4696 | -3.7615 |
| Bathrooms | 8.2509 | 10.2640 |
| Age | 0.4708 | 3.6624 |
| Distance | -16.1234 | -22.1799 |

### Which weights became zero?

In this experiment, **none** of Lasso weights became exactly **0**. Therefore, Lasso did **not** perform feature selection on this dataset with current value of `alpha = 0.1`.

### Why?

Lasso produces sparse solutions only when regularization strength is large enough to push less important coefficients exactly to zero. Here, `alpha = 0.1` was not strong enough, so every feature kept a non-zero coefficient.

Ridge also kept every coefficient non-zero, which is expected because L2 regularization shrinks weights but rarely makes them exactly zero.

### Why does L1 produce sparse solutions while L2 does not?

- **L1 (Lasso)** uses the penalty `α Σ|w|`. This penalty can drive unimportant coefficients exactly to **0**, automatically removing features from the model and producing a **sparse** solution.
- **L2 (Ridge)** uses the penalty `α Σw²`. This penalty continuously shrinks coefficients toward **0**, but they usually remain small rather than becoming exactly zero.

### Conclusion

For this experiment:

- Lasso reduced coefficient magnitudes but did **not** set any coefficient to zero.
- Ridge also reduced coefficient magnitudes and kept every feature.
- No sparse solution was observed because `alpha = 0.1` was too small for this dataset.
```

In [28]:
lasso_model2=LassoRegression(5,alpha=1)
ridge_model2=RidgeRegression(5,alpha=1,learning_rate=0.001)
lasso_model2.fit(X,y,epochs=1000)
ridge_model2.fit(X,y,epochs=1000)

print("\n  Lasso Weights",lasso_model2.weights)
print("\n  Ridge Weights",ridge_model2.weights)

Epoch    0 | Cost: 239990.0000 | L1 penalty: 0.0000
Epoch  200 | Cost: 756.1729 | L1 penalty: 41.6395
Epoch  400 | Cost: 690.7100 | L1 penalty: 42.7996
Epoch  600 | Cost: 634.0317 | L1 penalty: 43.9569
Epoch  800 | Cost: 588.2420 | L1 penalty: 46.7503
  Epoch    0 | Cost: 239990.0000 | L2 penalty: 0.0000
  Epoch  200 | Cost: 1382.5712 | L2 penalty: 651.6084
  Epoch  400 | Cost: 1361.3806 | L2 penalty: 669.1173
  Epoch  600 | Cost: 1352.9310 | L2 penalty: 684.2901
  Epoch  800 | Cost: 1349.4624 | L2 penalty: 695.4646

  Lasso Weights [24.820526124636494, -1.483597952771149, 7.5547375906481555, 0.2843323720146498, -15.763288702116478]

  Ridge Weights [24.16588886215188, 1.3840720742671218, 5.770015016053326, -3.222804577848057, -8.576900714207206]


**Working With House Pricing Dataset**

In [34]:
import pandas as pd
df = pd.read_csv("./house_pricing.csv")

In [35]:
df

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [37]:
y=df["price"]

In [38]:
df["area"]=df["area"]/df["area"].mean()
df

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,1.440625,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,1.739623,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,1.933777,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,1.456158,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,1.440625,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,0.582463,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,0.465970,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,0.702839,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,0.564989,3,1,1,no,no,no,no,no,0,no,furnished


In [40]:
X=df[["area","bedrooms","bathrooms","stories","parking"]]
X

,area,bedrooms,bathrooms,stories,parking
0,1.440625,4,2,3,2
1,1.739623,4,4,4,3
2,1.933777,3,2,2,2
3,1.456158,4,2,2,3
4,1.440625,4,1,2,2
...,...,...,...,...,...
540,0.582463,2,1,1,2
541,0.465970,3,1,1,0
542,0.702839,2,1,1,0
543,0.564989,3,1,1,0


In [41]:
from sklearn.model_selection import train_test_split

In [ ]:
lasso_model2=LassoRegression(5,alpha=1)
ridge_model2=RidgeRegression(5,alpha=1,learning_rate=0.001)

lasso_model2.fit(X,y,epochs=1000)
ridge_model2.fit(X,y,epochs=1000)

print("\n  Lasso Weights",lasso_model2.weights)
print("\n  Ridge Weights",ridge_model2.weights)

In [42]:
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.3)

In [50]:
# lasso_model3=LassoRegression(5,alpha=1,learning_rate=0.001)
# ridge_model3=RidgeRegression(5,alpha=1,learning_rate=0.001)

# lasso_model3.fit(X_train,y_train,epochs=1000)
# ridge_model3.fit(X_train,y_train,epochs=1000)

# print("\n  Lasso Weights",lasso_model3.weights)
# print("\n  Ridge Weights",ridge_model3.weights)

In [51]:
from sklearn.linear_model import Lasso , Ridge

In [56]:
lasso_sk=Lasso(alpha=10)
ridge_sk=Ridge(alpha=10)

lasso_sk.fit(X_train,y_train)
ridge_sk.fit(X_train,y_train)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",10
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' t

In [57]:
lasso_sk.coef_

array([1677603.62561727,   56280.61962305, 1183231.17103776,
        616616.24831746,  333267.83409142])

In [58]:
ridge_sk.coef_

array([1458476.54210453,   98193.44235916, 1086327.99186961,
        608779.76810797,  366804.41488372])